# Substitution Ciphers

In [ ]:
import pickle
from pathlib import Path
from math import gcd

import string

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
data_dir = Path().resolve().parents[0] / "data" # create a path to directory "data"

## Introduction


Substitution ciphers are classical encryption techniques in which each element of a plaintext message, typically a letter, is replaced by another symbol according to a fixed mapping defined by a **key**. 

The replacement remains consistent throughout the message.​

The receiver decrypts the message by applying the **inverse substitution**, recovering the original plaintext.

Although historically important, substitution ciphers are vulnerable to cryptanalytic methods such as **frequency analysis**, which exploit statistical patterns in natural language and also as **brute force attacks**, due to the limited key space.

## Caesar Cipher

The method is named after Julius Caesar, who used it in his private correspondence. Each letter in the plaintext is replaced by a letter shifted by some fixed number of positions down the alphabet. 

Its a extremely simple encryption rule, it has only 26 possibile keys!

![Caesar Encryption](images/caesar.png)

**Types of attack of Caesar Cipher:**

**Brute force**: ​The English alphabet is 26 letters long, meaning that only 26 shifts are possible. Hence, you can try all possibilities and check whether the resulting plaintext makes sense.​

### Encrypt

Encryption shifts each letter of the plaintext **forward** by a number of positions $k$ determined by the key.

In [ ]:
def caesar_encrypt(plaintext, shift=0):
    ''' Encrypt `plaintext` (str) as a caesar cipher with a given `shift` (int)
        and return the `ciphertext` (str) '''
    # define plaintext alphabet
    plain_alphabet = string.ascii_lowercase
    # define ciphertext alphabet
    cipher_alphabet = plain_alphabet[shift:] + plain_alphabet[:shift]
    # build a mapping that trasforms a plaintext letter into a ciphertext letter
    mapping = dict(zip(plain_alphabet, cipher_alphabet))
    # apply the transformation to the plaintext
    ciphertext = ''.join(mapping.get(char, char) for char in plaintext)
    return ciphertext

In [ ]:
# code snippet to test the implementation of the decoder
plaintext = 'hello!'
ciphertext = caesar_encrypt(plaintext, shift=4)

print(plaintext, '->', ciphertext) # expected output 'hello! -> lipps!'

### Decrypt

Decryption works by shifting each letter of the ciphertext **backward** by the same number of positions $k$ used during encryption. If the cipher used a shift of 𝑘, decryption applies a shift of $−k$ to recover the original plaintext.

In [ ]:
def caesar_decrypt(ciphertext, shift=0):
    ''' Decrypt `ciphertext` (str) as a caesar cipher with a given `shift` (int)
        and return the `plaintext` (str) '''
    # define plaintext alphabet
    plain_alphabet = string.ascii_lowercase
    # define ciphertext alphabet
    cipher_alphabet = plain_alphabet[shift:] + plain_alphabet[:shift]
    # build a mapping that trasform a plaintext letter into a ciphertext letter
    mapping = dict(zip(cipher_alphabet, plain_alphabet))
    # apply the transformation to the plaintext
    plaintext = ''.join(mapping.get(char, char) for char in ciphertext)
    return plaintext

In [ ]:
# code snippet to test the implementation of the decoder
ciphertext = 'lipps!' # 'hello!' encoded with shift=4
plaintext = caesar_decrypt(ciphertext, shift=4)

print(ciphertext, '->', plaintext)

### Ciphertext

In [ ]:
# Load ciphertext

with open(data_dir / 'ciphertext_caesar.txt', mode='r', encoding='utf-8') as f:
    ciphertext = f.read()

print(ciphertext[:300])


### Brute Force

A **brute force attack** consists of trying **all possible keys** (i.e., shifts) of the Caesar cipher.  
Since the alphabet has **26 letters**, there are **25 possible keys** (let us remove the null shift which keep the plaintext plain).

The attacker decrypts the ciphertext using each possible shift and checks which result produces a **meaningful plaintext message**.

In [ ]:
# iterate over all possible 26 shift values (let us include also 0)
for i in range(26):
    print(i, caesar_decrypt(ciphertext, i)[:70])

In [ ]:
# select the right shift
shift = 18

# decrypt ciphertext
plaintext = caesar_decrypt(ciphertext, shift)
print(plaintext) # print only the first 100 chars

## Simple Substitution Cipher

A Simple Substitution Cipher replaces each plaintext character with a **different ciphertext character**. As in the Caesar cipher, the plaintext and ciphertext use the **same alphabet**.

The mapping between plaintext letters and ciphertext letters can be **any permutation of the alphabet**, giving $26! \approx 10^{26} \approx 2^{88}$ possible substitution keys.

The receiver **decrypts** the text by performing the inverse substitution.

![Simple Substitution](images/simple_substitution.png)

**Type of attacks of a Substitution Cipher:**

1) **Brute Force**: consists of trying every possible key (i.e., shift). Even if we assume that each attempt takes only 1 nanosecond, checking all keys would require **more than $10^9$ years**. Since the number of possible substitution keys is $26! \approx 10^26$, it is not feasible for modern computers to test all possibilities.

2) **Frequency Analysis**: A substitution cipher keeps the **statistical characteristics of the language**, such as how often each letter appears. This property allows an attacker to estimate the plaintext by studying the frequency of letters in the ciphertext.

For sufficiently long ciphertexts (so that the statistics are meaningful), a possible approach is:
- Replace the **most frequent letter in the ciphertext** with the **most frequent letter of the language**
- Replace the **second most frequent ciphertext letter** with the **second most frequent letter**
- Continue this process for the remaining letters

However, the observed frequencies in the ciphertext will not perfectly match the expected probabilities of the language. As a result, the automatic substitutions will only partially recover the correct mapping of the substitution cipher, and additional manual adjustments are usually required to obtain the correct plaintext.

### Encryption

In [ ]:
def simple_encrypt(plaintext, mapping):
    ''' Encrypt `plaintext` (str) as a simple substitution cipher with a given
       `mapping` (dict) from plaintext letters to ciphertext letters '''
    ciphertext = ''.join(mapping.get(char, char) for char in plaintext)
    return ciphertext

In [ ]:
# code snippet to test the implementation of the encryption function
plaintext = 'hello!'
mapping = {'h': 'a', 'e': 'p', 'l': 'w', 'o': 'q'}

ciphertext = simple_encrypt(plaintext, mapping)

print(plaintext, '->', ciphertext) # expected output 'hello! -> apwwq!'

### Decryption

In [ ]:
def simple_decrypt(ciphertext, mapping):
    ''' Decrypt `ciphertext` (str) as a simple substitution cipher with a given
        `mapping` (dict) from plaintext letters to ciphertext letters '''
    inv_mapping = dict(zip(mapping.values(), mapping.keys()))
    plaintext = ''.join(inv_mapping.get(char, char) for char in ciphertext)
    return plaintext

In [ ]:
# code snippet to test the implementation of the decryption function
mapping = {'h': 'a', 'e': 'p', 'l': 'w', 'o': 'q'}  # previous mapping
ciphertext = 'apwwq!'

plaintext = simple_decrypt(ciphertext, mapping)

print(ciphertext, '->', plaintext)  # expected output 'apwwq! -> hello!'

### Ciphertext

In [ ]:
# Load ciphertext
with open(data_dir / "ciphertext_simple.txt", mode='r', encoding='utf-8') as f:
    ciphertext = f.read()

print(ciphertext[:500])


### Frequency Analysis Attack

#### English Letters Distribution

In [ ]:
def letter_distribution(text):
    ''' Return the `distribution` (dict) of the letters in `text` (str) '''
    alphabet = string.ascii_lowercase
    hist = np.empty(len(alphabet), dtype=int)
    for i, letter in enumerate(alphabet):
        hist[i] = text.lower().count(letter)
    hist = map(float, hist / np.sum(hist))
    distribution = dict(zip(alphabet, hist))
    return distribution

In [ ]:
# code snippet to test the implementation of `letter_distribution`
text = 'hello world!'

letter_distribution(text)
# expected ouput:
# {'d': 0.1, 'e': 0.1, 'h': 0.1, 'l': 0.3, 'o': 0.2, 'r': 0.1, 'w': 0.1, ...}

In [ ]:
# Load text from which we estimate the English letter distribution
with open(data_dir / "wikipedia_cybersecurity.txt", mode='r', encoding='utf-8') as f:
    text = f.read()

print(text[:500])

In [ ]:
eng_distrib = letter_distribution(text)
eng_distrib

In [ ]:
def plot_distribution(distribution, ax=None, title=None):
    ''' plot a letter distribution'''
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(np.arange(len(distribution)), list(distribution.values()), 1)
    ax.set(xticks=np.arange(len(distribution)), xticklabels=distribution.keys())
    ax.set(xlabel='letter', ylabel='probability', title=title)
    ax.grid()

In [ ]:
plot_distribution(eng_distrib, title='English letter distribution')

#### Attack

In [ ]:
# load ciphertext
with open(data_dir / "ciphertext_simple.txt", mode='r', encoding='utf-8') as f:
    ciphertext = f.read()

print(ciphertext[:500])

In [ ]:
# Attack